# CBIS-DDSM: **ROI-cropped** Lesion Segmentation (high-Dice)

Full-image segmentation capped at mean Dice ~0.32 because lesions are ~1% of the pixels. Here we
segment on **lesion-centered crops** taken from the **original full-resolution** JPEG (bbox from
the ROI mask, expanded x1.8 for context), resized to 256 - so the lesion fills ~30% of the frame
and keeps its detail. Same fine-tuned MobileNetV2 Attention U-Net (v3 recipe).

This answers *"how precisely can you delineate the lesion given its region"* (assumes a detected
ROI) - the standard route to Dice ~0.6-0.8. One crop per abnormality (per-ROI), patient-disjoint.


In [ ]:
# ---- GPU: pin to RTX ----
import os
os.environ.setdefault('DML_VISIBLE_DEVICES', '0')
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPUs:', gpus)
if gpus:
    try:
        tf.config.set_visible_devices([gpus[0]], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except Exception as e:
        print('gpu cfg:', e)
print('Visible:', tf.config.get_visible_devices('GPU'))

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2

In [ ]:
# ============================================================ CONFIG
DATA_ROOT  = 'D:/cbis-ddsm'; JPEG_ROOT = DATA_ROOT + '/jpeg'; CSV_DIR = DATA_ROOT + '/csv'
DICOM_INFO = CSV_DIR + '/dicom_info.csv'; SAVE_DIR = 'G:/My Drive/VS_CC/models'
CACHE = 'D:/cbis-ddsm/_seg_cache'
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
TRAIN_CSVS = ['mass_case_description_train_set.csv', 'calc_case_description_train_set.csv']
TEST_CSVS  = ['mass_case_description_test_set.csv',  'calc_case_description_test_set.csv']
SZ = 256; BATCH = 4; RANDOM_STATE = 2021; MASK_THRESH = 127; IMG_SIZE = SZ
MARGIN = 1.8                       # crop = lesion bbox expanded by this factor
WARMUP_EPOCHS, FT_EPOCHS = 12, 40
LR_WARMUP, LR_FT = 1e-3, 1e-4
TV_ALPHA, TV_BETA, TV_GAMMA = 0.7, 0.3, 0.75

## 1. Resolve full image + ROI mask (dicom_info.csv)

In [ ]:
def _to_local(p):
    p = str(p).replace('\\', '/'); i = p.find('jpeg/')
    return os.path.join(JPEG_ROOT, p[i + 5:]) if i >= 0 else p
def build_uid_maps(dinfo):
    di = pd.read_csv(dinfo); di['SeriesDescription'] = di['SeriesDescription'].fillna('')
    di['local'] = di['image_path'].apply(_to_local); di['uid'] = di['SeriesInstanceUID'].astype(str)
    full, mask, allf = {}, {}, {}
    for _, r in di.iterrows():
        allf.setdefault(r['uid'], []).append(r['local']); d = r['SeriesDescription'].lower()
        if 'full' in d: full.setdefault(r['uid'], r['local'])
        elif 'roi mask' in d: mask.setdefault(r['uid'], r['local'])
    return full, mask, allf
FULL_MAP, MASK_MAP, ANY_MAP = build_uid_maps(DICOM_INFO)
def series_uid(p):
    q = str(p).replace('\\', '/').strip().split('/'); return q[-2] if len(q) >= 2 else None
def is_mask_like(g):
    return g is not None and np.mean((g <= 10) | (g >= 245)) > 0.90 and len(np.unique(g)) <= 64
def resolve_mask(p):
    u = series_uid(p)
    if u in MASK_MAP: return MASK_MAP[u]
    for c in ANY_MAP.get(u, []):
        if is_mask_like(cv2.imread(c, cv2.IMREAD_GRAYSCALE)): return c
    return None
def load_tables(names):
    df = pd.concat([pd.read_csv(os.path.join(CSV_DIR, n)) for n in names if os.path.exists(os.path.join(CSV_DIR, n))], ignore_index=True)
    df.columns = [c.strip().replace(' ', '_').lower() for c in df.columns]; return df
def prep(df):
    df['full_path'] = df['image_file_path'].apply(lambda p: FULL_MAP.get(series_uid(p)))
    df['mask_path'] = df['roi_mask_file_path'].apply(resolve_mask)
    df['patient_id'] = df['patient_id'].astype(str)
    return df[['patient_id', 'full_path', 'mask_path']].dropna(subset=['full_path', 'mask_path'])
train_df = prep(load_tables(TRAIN_CSVS)); test_df = prep(load_tables(TEST_CSVS))
ov = set(train_df.patient_id) & set(test_df.patient_id)
if ov: train_df = train_df[~train_df.patient_id.isin(ov)].reset_index(drop=True)
print('per-ROI rows -> train', len(train_df), 'test', len(test_df),
      'disjoint', set(train_df.patient_id).isdisjoint(set(test_df.patient_id)))

## 2. ROI crop from full-res JPEG + cropped memmap cache

In [ ]:
_clahe = cv2.createCLAHE(2.0, (8, 8))
def crop_around(fp, mp, margin=MARGIN, out=SZ):
    img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)                 # full resolution
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE) if mp else None
    if img is None or not is_mask_like(m):
        return None, None
    if m.shape != img.shape:
        m = cv2.resize(m, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    m = (m >= MASK_THRESH).astype(np.uint8)
    ys, xs = np.where(m)
    if len(ys) == 0: return None, None
    r0, r1, c0, c1 = ys.min(), ys.max(), xs.min(), xs.max()
    cy, cx = (r0 + r1) // 2, (c0 + c1) // 2
    half = int(max(r1 - r0, c1 - c0) * margin / 2) + 8
    H, W = img.shape
    ri0, ri1 = max(0, cy - half), min(H, cy + half); ci0, ci1 = max(0, cx - half), min(W, cx + half)
    ci = img[ri0:ri1, ci0:ci1]; cm = m[ri0:ri1, ci0:ci1]
    if ci.size == 0: return None, None
    ci = _clahe.apply(cv2.resize(ci, (out, out)))
    cm = (cv2.resize(cm * 255, (out, out), interpolation=cv2.INTER_NEAREST) >= 127).astype(np.uint8)
    return np.stack([ci] * 3, -1), cm

def make_items(df):
    return [(r.full_path, r.mask_path) for r in df.itertuples(index=False)], df['patient_id'].values
items_tr, pats_tr = make_items(train_df); items_te, pats_te = make_items(test_df)

def _p(prefix, ch): return os.path.join(CACHE, f'{prefix}_{ch}_{SZ}.npy')
def build_cache(items, prefix):
    if os.path.exists(_p(prefix, 'valid')):
        print(f'  cache {prefix} exists -> skip'); return
    n = len(items)
    X = np.lib.format.open_memmap(_p(prefix, 'X'), mode='w+', dtype=np.uint8, shape=(n, SZ, SZ, 3))
    Y = np.lib.format.open_memmap(_p(prefix, 'Y'), mode='w+', dtype=np.uint8, shape=(n, SZ, SZ, 1))
    valid = np.zeros(n, bool)
    for i, (fp, mp) in enumerate(items):
        ci, cm = crop_around(fp, mp)
        if ci is not None:
            X[i] = ci; Y[i] = cm[..., None]; valid[i] = True
        if (i + 1) % 400 == 0: print('   cropped', i + 1)
    X.flush(); Y.flush(); del X, Y; np.save(_p(prefix, 'valid'), valid)
    print(f'  {prefix}: {valid.sum()}/{n} valid')
print('caching TRAIN crops...'); build_cache(items_tr, 'roitr')
print('caching TEST crops...');  build_cache(items_te, 'roite')
Xtr = np.load(_p('roitr', 'X'), mmap_mode='r'); Ytr = np.load(_p('roitr', 'Y'), mmap_mode='r'); vtr = np.load(_p('roitr', 'valid'))
Xte = np.load(_p('roite', 'X'), mmap_mode='r'); Yte = np.load(_p('roite', 'Y'), mmap_mode='r'); vte = np.load(_p('roite', 'valid'))
idx_tr = np.where(vtr)[0]; pats_tr_v = pats_tr[vtr]
print('crops -> train valid', len(idx_tr), '| test valid', int(vte.sum()), '| lesion coverage %.3f' % float(np.asarray(Ytr[idx_tr[:100]]).mean()))

class SegSeq(tf.keras.utils.Sequence):
    def __init__(self, X, Y, idx, batch, shuffle=True, augment=False):
        self.X = X; self.Y = Y; self.idx = np.asarray(idx); self.batch = batch
        self.shuffle = shuffle; self.augment = augment; self.on_epoch_end()
    def __len__(self): return int(np.ceil(len(self.idx) / self.batch))
    def on_epoch_end(self):
        self.order = np.arange(len(self.idx))
        if self.shuffle: np.random.shuffle(self.order)
    def __getitem__(self, b):
        sel = np.sort(self.idx[self.order[b*self.batch:(b+1)*self.batch]])
        X = self.X[sel].astype('float32'); Y = self.Y[sel].astype('float32')
        if self.augment:
            for k in range(len(X)):
                if np.random.rand() < 0.5: X[k] = X[k, :, ::-1]; Y[k] = Y[k, :, ::-1]
                if np.random.rand() < 0.5: X[k] = X[k, ::-1]; Y[k] = Y[k, ::-1]
                r = np.random.randint(4)
                if r: X[k] = np.rot90(X[k], r); Y[k] = np.rot90(Y[k], r)
        return X, Y

### 2b. DIAGNOSTIC - cropped patch + lesion mask

In [ ]:
vi = np.where(vte)[0][:4]
fig, ax = plt.subplots(1, 4, figsize=(12, 3.2), dpi=110)
for a, idx in zip(ax, vi):
    a.imshow(np.asarray(Xte[idx])[..., 0], cmap='gray'); a.contour(np.asarray(Yte[idx, :, :, 0]), colors='lime', linewidths=1.3)
    a.axis('off')
plt.tight_layout(); plt.show()
print('lesion should be roughly centered and fill a good fraction of the crop')

## 3. MobileNetV2-encoder Attention U-Net (fine-tunable)

In [ ]:
def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    return x
def attn_gate(skip, gating, inter):
    tx = layers.Conv2D(inter, 1)(skip); pg = layers.Conv2D(inter, 1)(gating)
    r = layers.Activation('relu')(layers.Add()([tx, pg]))
    psi = layers.Conv2D(1, 1, activation='sigmoid')(r)
    return layers.Multiply()([skip, psi])
NAMES = ['block_1_expand_relu', 'block_3_expand_relu', 'block_6_expand_relu', 'block_13_expand_relu', 'block_16_project']
def build_model(sz=SZ):
    inp = layers.Input((sz, sz, 3)); x = layers.Rescaling(1./127.5, offset=-1)(inp)
    base = MobileNetV2(input_tensor=x, include_top=False, weights='imagenet'); base.trainable = False
    s128, s64, s32, s16, s8 = [base.get_layer(n).output for n in NAMES]
    def up(x, skip, f):
        x = layers.UpSampling2D()(x); a = attn_gate(skip, x, f); return conv_block(layers.Concatenate()([x, a]), f)
    d = up(s8, s16, 256); d = up(d, s32, 128); d = up(d, s64, 64); d = up(d, s128, 32)
    d = conv_block(layers.UpSampling2D()(d), 16)
    return Model(inp, layers.Conv2D(1, 1, activation='sigmoid')(d)), base
def unfreeze_encoder(base):
    base.trainable = True
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization): l.trainable = False
    print('encoder conv layers unfrozen (BN frozen)')

## 4. BCE + Focal-Tversky loss + metrics

In [ ]:
def _flat(y_true, y_pred):
    return tf.reshape(tf.cast(y_true, tf.float32), [-1]), tf.reshape(y_pred, [-1])
def tversky_index(y_true, y_pred, a=TV_ALPHA, b=TV_BETA, smooth=1.):
    yt, yp = _flat(y_true, y_pred)
    tp = tf.reduce_sum(yt*yp); fn = tf.reduce_sum(yt*(1-yp)); fp = tf.reduce_sum((1-yt)*yp)
    return (tp + smooth) / (tp + a*fn + b*fp + smooth)
def bce_ftversky(y_true, y_pred):
    yt = tf.cast(y_true, tf.float32)
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(yt, y_pred)) + tf.pow(1 - tversky_index(yt, y_pred), TV_GAMMA)
def dice_coef(y_true, y_pred, smooth=1.):
    yt, yp = _flat(y_true, y_pred); inter = tf.reduce_sum(yt*yp)
    return (2*inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)
def iou_metric(y_true, y_pred, smooth=1.):
    yt, yp = _flat(y_true, tf.cast(y_pred > 0.5, tf.float32)); inter = tf.reduce_sum(yt*yp)
    return (inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) - inter + smooth)
model, base = build_model()
model.compile(Adam(LR_WARMUP), loss=bce_ftversky, metrics=[dice_coef, iou_metric])
print('params %.2fM' % (model.count_params()/1e6))

## 5. Two-phase training

In [ ]:
gss = GroupShuffleSplit(1, test_size=0.15, random_state=RANDOM_STATE)
lo, hi = next(gss.split(idx_tr, groups=pats_tr_v))
train_gen = SegSeq(Xtr, Ytr, idx_tr[lo], BATCH, shuffle=True, augment=True)
val_gen   = SegSeq(Xtr, Ytr, idx_tr[hi], BATCH, shuffle=False, augment=False)
print('fit', len(lo), 'val', len(hi))
print('=== Phase 1: warm up decoder (encoder frozen) ===')
model.fit(train_gen, validation_data=val_gen, epochs=WARMUP_EPOCHS,
          callbacks=[EarlyStopping(monitor='val_dice_coef', mode='max', patience=6, restore_best_weights=True)], verbose=2)
print('\n=== Phase 2: fine-tune encoder (LR=%g) ===' % LR_FT)
unfreeze_encoder(base)
model.compile(Adam(LR_FT), loss=bce_ftversky, metrics=[dice_coef, iou_metric])
model.fit(train_gen, validation_data=val_gen, epochs=FT_EPOCHS,
          callbacks=[EarlyStopping(monitor='val_dice_coef', mode='max', patience=12, restore_best_weights=True),
                     ReduceLROnPlateau(monitor='val_dice_coef', mode='max', factor=0.5, patience=5, min_lr=1e-6)], verbose=2)

## 6. Evaluate on the patient-disjoint test crops

In [ ]:
P = model.predict(Xte, batch_size=BATCH, verbose=0)[..., 0]
def per_image(y, p, sel, thr=0.5):
    d, io, hit = [], [], []
    for i in sel:
        gt = np.asarray(y[i, :, :, 0]).astype(bool); pm = p[i] >= thr
        inter = np.logical_and(gt, pm).sum(); s = gt.sum() + pm.sum(); u = np.logical_or(gt, pm).sum()
        d.append(2*inter/s if s else 1.0); io.append(inter/u if u else 1.0)
        yy, xx = np.unravel_index(int(np.argmax(p[i])), p[i].shape); hit.append(int(gt[yy, xx]))
    return np.array(d), np.array(io), np.array(hit)
d, io, hit = per_image(Yte, P, np.where(vte)[0])
print(f'TEST (ROI crops)  Dice {d.mean():.3f} +/- {d.std():.3f} | IoU {io.mean():.3f} | pointing-game {hit.mean()*100:.1f}%  (n={len(d)})')
print(f'   median Dice {np.median(d):.3f} | Dice>0.5 rate {np.mean(d > 0.5)*100:.1f}% | Dice>0.7 rate {np.mean(d > 0.7)*100:.1f}%')

### 6b. Overlays: crop + GT (green) + prediction (red)

In [ ]:
sel = np.where(vte)[0]; order = sel[np.argsort(-d)][:6]
fig, ax = plt.subplots(1, 6, figsize=(18, 3.2), dpi=110)
dd = dict(zip(sel, d))
for a, idx in zip(ax, order):
    a.imshow(np.asarray(Xte[idx])[..., 0], cmap='gray')
    a.contour(np.asarray(Yte[idx, :, :, 0]), colors='lime', linewidths=1.2)
    a.contour((P[idx] >= 0.5).astype(float), colors='red', linewidths=1.0)
    a.set_title(f'Dice {dd[idx]:.2f}', fontsize=9); a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'cbis_roi_seg_overlays.png'), dpi=200, bbox_inches='tight'); plt.show()

## 7. Summary + save

In [ ]:
summary = (
    f"ROI-cropped lesion segmentation (lesion-centered crops from full-res JPEG, margin x{MARGIN}, "
    f"resized {SZ}x{SZ}) with a two-phase fine-tuned MobileNetV2 Attention U-Net (BCE+Focal-Tversky, aug), "
    f"CBIS-DDSM patient-disjoint. Test: Dice {d.mean():.3f} +/- {d.std():.3f} (median {np.median(d):.3f}), "
    f"IoU {io.mean():.3f}, pointing-game {hit.mean()*100:.1f}%, Dice>0.5 {np.mean(d > 0.5)*100:.1f}%, "
    f"Dice>0.7 {np.mean(d > 0.7)*100:.1f}% over {len(d)} test lesions. "
    f"(Full-image v3 was Dice 0.316/PG 54%; weakly-supervised ~0-5% PG.)"
)
print(summary)
open(os.path.join(SAVE_DIR, 'cbis_roi_seg_summary.txt'), 'w').write(summary)
model.save_weights(os.path.join(SAVE_DIR, 'cbis_roi_seg.weights.h5'))
print('\nSaved to', SAVE_DIR)